# LLM과 Tool


## LLM 준비


In [ ]:
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool

# .env 파일 로드 및 API 키 확인
load_dotenv()

# OpenAI LLM 준비
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

## 데코레이터를 통한 Tool 구성


In [ ]:
## 단계 - 도구 구성
@tool
def calculator(num1: float, num2: float, operation: str) -> float:
    """두 숫자와 연산자를 받아 계산 결과를 반환하는 도구 함수"""

    print(
        f"===> Calculator 도구 호출: num1={num1}, num2={num2}, operation='{operation}'"
    )

    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        if num2 != 0:
            return num1 / num2
        else:
            return "Error: Division by zero"
    else:
        return "Error: Invalid operation"

## LLM에 도구(tools) 연결하기


In [ ]:
# 도구 목록
tools = [calculator]

# 도구 이름 → 도구 함수 매핑
tool_map = {t.name: t for t in tools}

# 도구 목록을 LLM에 연결
llm_with_tools = llm.bind_tools(tools)

## 프롬프트 생성

도구를 사용할 수 있음을 알려주고, 도구 사용이 필요한 메시지를 요청


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 계산을 도와주는 유용한 도구를 가지고 있습니다. 필요한 경우 도구를 사용하여 계산을 수행하세요.",
        ),
        ("human", "{question}"),
    ]
)

prompt_messages = prompt.format_messages(question="23 곱하기 3은 얼마인가요?")

## 도구가 연결된 LLM과 대화 처리 과정

1. 도구 사용이 필요한 프롬프트 메시지를 LLM에 전달
2. LLM이 사용자 요청을 분석하여 특정 도구 호출이 필요한 경우 AIMessage에 호출될 도구 정보를 전달(tool_calls)
3. 프로그램에서 Tool을 실행하고 그 결과(ToolMessage)를 다시 LLM에 전달하여 최종 답변을 생성


In [ ]:
from langchain_core.messages import ToolMessage

ai_message = llm_with_tools.invoke(prompt_messages)
print("===> 도구 호출 요청:", ai_message.tool_calls)

# 기존 메시지 + 도구 실행 요청 메시지
messages = prompt_messages + [ai_message]

# 요청된 도구 함수 실행 후 ToolMessage 추가
for tool_call in ai_message.tool_calls:
    print(f"===> 도구 실행: {tool_call['name']} with args {tool_call['args']}")
    tool_fn = tool_map[tool_call["name"]]
    result = tool_fn.invoke(tool_call["args"])
    print(f"===> 도구 실행 결과: {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

# 모든 메시지를 LLM에 전달 → 최종 답변 생성
final_ai_message = llm_with_tools.invoke(messages)
print("LLM 최종 답변:", final_ai_message.content)

messages += [final_ai_message]
messages

## Agent에 도구 연결하고 실행하기

Agent 실행: `AIMessage(tool_calls)` - `tool 함수 실행` - `ToolMessage` 과정이 내부적으로 진행됨.


In [ ]:
# 에이전트 생성

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
)

response = agent.invoke({"messages": prompt_messages})

print("에이전트 응답:", response["messages"][-1].content)
response["messages"]